# SupremeAI - Llama 3.3-70B Inference Server

Runs Llama 3.3-70B-Instruct on Colab T4 GPU and serves OpenAI-compatible API for SupremeAI.

### Requirements:
1. Runtime > Change runtime type > T4 GPU ✓
2. Accept Llama model terms - https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct ✓
3. HuggingFace Token - https://huggingface.co/settings/tokens

### ⚠️ Warnings:
- Model is ~35GB compressed - loading takes 3-5 minutes
- Keep notebook tab open or tunnel will disconnect
- Colab free tier sometimes disconnects in 12-24 hours


In [ ]:
# Step 1: Install dependencies (Colab-optimized)
!pip install -q 'numpy<2.1' --force-reinstall
!pip install -q --upgrade transformers accelerate bitsandbytes requests flask flask-cors

# AirLLM + Optimum are optional; install if possible but don't block
!pip install -q --no-deps airllm optimum 2>/dev/null || true

# Cloudflare Tunnel (optional - if download fails, just continue)
!curl -L --output cloudflared.tgz https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.tgz 2>/dev/null && tar -xzf cloudflared.tgz 2>/dev/null && chmod +x cloudflared || echo 'Cloudflare Tunnel (optional, skipped if failed)'

import torch
import sys
print('\n' + '='*70)
print('Dependencies configured for Colab')
print('='*70)
print(f'  Python: {sys.version.split()[0]}')
print(f'  PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')
else:
    print('  WARNING: NO GPU - Change Runtime to T4/A100')
print('='*70)

In [ ]:
# Step 2: Configure HF Token + Request Gated Model Access
import os, shutil

# ========== SETUP INSTRUCTIONS ==========
# STEP A: Accept the license on Hugging Face (REQUIRED)
#   1. Open: https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct
#   2. Click the blue 'Agree and access repository' button (if you see it)
#   3. Accept the Meta license agreement
#   4. Refresh the page - you should now see the 'Request access' button
#   5. Click 'Request access', fill form (takes 1-2 minutes for approval)

# STEP B: Create a Hugging Face API token
#   1. Go to: https://huggingface.co/settings/tokens
#   2. Click 'New token'
#   3. Name it 'supremeai-llama'
#   4. Choose 'Read' access (fine-grained access to model repos)
#   5. Create and copy the token (looks like: hf_aBcDeFgHiJkLmNoPqRsTuVwXyZ123456)

# STEP C: Paste your token below (entire string starting with hf_)
HF_TOKEN = 'hf_PASTE_YOUR_TOKEN_HERE'

MODEL_ID = 'meta-llama/Llama-3.3-70B-Instruct'
COMPRESSION = '4bit'
MAX_NEW_TOKENS = 512
PORT = 8081

# Validate token format
if 'PASTE_YOUR_TOKEN' in HF_TOKEN:
    raise RuntimeError('Token not set. Follow STEP B above and paste your token here.')
if not HF_TOKEN.startswith('hf_') or len(HF_TOKEN) < 30:
    raise RuntimeError(f'Token looks invalid. Got: {HF_TOKEN[:20]}... (should be hf_...)')

os.environ['HF_TOKEN'] = HF_TOKEN
cache_dir = os.path.expanduser('~/.cache/huggingface/hub')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir, ignore_errors=True)

print('\n' + '='*70)
print('HF Token configured')
print('='*70)
print(f'Model: {MODEL_ID}')
print(f'Token: {HF_TOKEN[:15]}...')
print('Next: Run the diagnostic cell below to verify access')
print('='*70)

In [ ]:
# Step 2b: Verify Access to Gated Model (DIAGNOSTIC)
import requests

print('\n' + '='*70)
print('Verifying Hugging Face access...')
print('='*70)

# Check 1: Token validity
try:
    resp = requests.get(
        'https://huggingface.co/api/user',
        headers={'Authorization': f'Bearer {HF_TOKEN}'},
        timeout=10
    )
    if resp.status_code == 401:
        raise RuntimeError('Token is invalid/expired. Go to https://huggingface.co/settings/tokens and create a new one.')
    elif resp.status_code != 200:
        raise RuntimeError(f'HF API error: {resp.status_code}')
    
    user = resp.json()
    print(f'✓ Token is valid (user: {user.get("name", "unknown")})')
except Exception as e:
    raise RuntimeError(f'Token check failed: {e}')

# Check 2: Access to Llama 3.3-70B (gated model)
resp = requests.get(
    f'https://huggingface.co/api/models/{MODEL_ID}',
    headers={'Authorization': f'Bearer {HF_TOKEN}'},
    timeout=10
)

if resp.status_code == 401 or resp.status_code == 403:
    print(f'✗ NO ACCESS to {MODEL_ID}')
    print('  Your account has not been approved for this gated model.')
    print(f'  FIX: Go to https://huggingface.co/{MODEL_ID}')
    print('       - Click "Agree and access repository"')
    print('       - Accept the Meta license')
    print('       - Click "Request access" and wait 1-2 minutes for approval')
    print('       - Then rerun this cell')
    raise RuntimeError('Gated model access not approved')
elif resp.status_code == 200:
    print(f'✓ Access to {MODEL_ID} confirmed')
else:
    raise RuntimeError(f'Unexpected HF response: {resp.status_code}')

print('='*70)
print('Ready to proceed to Step 3')

In [ ]:
# Step 3: Load Llama 3.3-70B model (robust auth + fallback)
import time
import traceback
import requests

print('\n' + '='*70)
print('Loading Llama 3.3-70B model')
print('='*70)
print('This can take several minutes on first run')
print('Download size: ~35GB')
print('='*70 + '\n')

# Validate token format
if not (isinstance(HF_TOKEN, str) and HF_TOKEN.startswith('hf_') and len(HF_TOKEN) > 10):
    raise RuntimeError('HF_TOKEN is missing/invalid. Put a valid hf_ token in Step 2.')

# Fast pre-check for gated access (avoids huggingface_hub login import issues)
check_url = f'https://huggingface.co/api/models/{MODEL_ID}'
resp = requests.get(check_url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
if resp.status_code == 401 or resp.status_code == 403:
    raise RuntimeError(
        'No access to gated model. Request access and accept license at '
        f'https://huggingface.co/{MODEL_ID} then rerun.'
    )
elif resp.status_code != 200:
    raise RuntimeError(f'Hugging Face API check failed: {resp.status_code} {resp.text[:200]}')

start = time.time()
model = None

# AirLLM is optional; if unavailable, transformers path is used.
try:
    print('Attempting AirLLM load...')
    from airllm import AutoModel
    kw = {'compression': COMPRESSION}
    kw['hf_token'] = HF_TOKEN
    model = AutoModel.from_pretrained(MODEL_ID, **kw)
    elapsed = time.time() - start
    print(f'\nAirLLM loaded in {elapsed:.0f}s ({elapsed/60:.1f} min)')

except Exception as e:
    print(f'AirLLM failed: {type(e).__name__}: {str(e)[:160]}')
    print('\nFalling back to transformers...')

    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        quantization_config = None
        if COMPRESSION == '4bit':
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type='nf4',
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            )

        print('Loading tokenizer...')
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

        print('Downloading model weights (this may take several minutes)...')
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            token=HF_TOKEN,
            device_map='auto',
            torch_dtype=torch.float16,
            quantization_config=quantization_config,
        )
        model.tokenizer = tokenizer
        elapsed = time.time() - start
        print(f'\nTransformers loaded in {elapsed:.0f}s ({elapsed/60:.1f} min)')
    except Exception as inner:
        print('\nTransformers fallback failed.')
        print(f'{type(inner).__name__}: {inner}')
        traceback.print_exc(limit=2)
        raise

print('='*70)
print(f'Model ready: {MODEL_ID}')
print('='*70)

In [ ]:
# Step 4: Start Flask API + Cloudflare Tunnel
import uuid, time, torch, subprocess, threading, sys
from datetime import datetime

# Self-heal optional web dependencies in live Colab kernel
try:
    from flask import Flask, request, jsonify
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'flask==3.0.0'])
    from flask import Flask, request, jsonify

try:
    from flask_cors import CORS
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'flask-cors==4.0.0'])
    from flask_cors import CORS

app = Flask(__name__)
CORS(app)
stats = {'req': 0, 'err': 0, 'start': time.time()}

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'healthy',
        'provider': 'cloudflare-tunnel',
        'model_active': MODEL_ID,
        'model_name': 'Llama 3.3-70B-Instruct',
        'compression': COMPRESSION,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'uptime': int(time.time() - stats['start']),
        'requests_total': stats['req'],
        'errors_total': stats['err']
    })

@app.route('/v1/models', methods=['GET'])
def list_models():
    return jsonify({
        'object': 'list',
        'data': [{
            'id': MODEL_ID,
            'object': 'model',
            'owned_by': 'supremeai-cloudflare',
            'name': 'Llama 3.3-70B-Instruct'
        }]
    })

@app.route('/v1/chat/completions', methods=['POST'])
def chat():
    try:
        stats['req'] += 1
        data = request.json or {}
        msgs = data.get('messages', [])
        mt = min(data.get('max_tokens', MAX_NEW_TOKENS), MAX_NEW_TOKENS)

        parts = []
        for m in msgs:
            r, c = m.get('role', 'user'), m.get('content', '')
            if r == 'system':
                parts.append('[INST] <<SYS>>\n' + c + '\n<</SYS>>\n')
            elif r == 'user':
                parts.append('[INST] ' + c + ' [/INST]')
            elif r == 'assistant':
                parts.append(c)

        prompt = '\n'.join(parts) if parts else '[INST] Hello [/INST]'

        if hasattr(model, 'tokenizer'):
            tokenizer = model.tokenizer
        else:
            from transformers import AutoTokenizer
            tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

        inputs = tokenizer(prompt, return_tensors='pt')
        if torch.cuda.is_available():
            inputs = {k: v.to(model.device) if hasattr(model, 'device') else v.cuda() for k, v in inputs.items()}

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=mt,
                do_sample=True,
                temperature=float(data.get('temperature', 0.7)),
                top_p=float(data.get('top_p', 0.95)),
                pad_token_id=tokenizer.eos_token_id
            )

        text = tokenizer.decode(output[0], skip_special_tokens=True)
        reply = text[len(prompt):].strip() if text.startswith(prompt) else text.strip()

        return jsonify({
            'id': 'chatcmpl-' + str(uuid.uuid4())[:8],
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': MODEL_ID,
            'choices': [{
                'index': 0,
                'message': {'role': 'assistant', 'content': reply},
                'finish_reason': 'stop'
            }],
            'usage': {
                'prompt_tokens': int(inputs['input_ids'].shape[1]),
                'completion_tokens': int(output.shape[1] - inputs['input_ids'].shape[1]),
                'total_tokens': int(output.shape[1])
            }
        })
    except Exception as e:
        stats['err'] += 1
        return jsonify({'error': str(e)}), 500

def run_flask():
    app.run(host='0.0.0.0', port=PORT)

threading.Thread(target=run_flask, daemon=True).start()
time.sleep(3)

print('\n' + '='*70)
print('Flask API started')
print(f'Local URL: http://127.0.0.1:{PORT}')

# Start Cloudflare Tunnel if binary exists
public_url = None
if subprocess.run(['bash', '-lc', 'test -x ./cloudflared'], capture_output=True).returncode == 0:
    tunnel = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    def read_tunnel():
        global public_url
        started = time.time()
        while time.time() - started < 30:
            line = tunnel.stderr.readline()
            if not line:
                continue
            if 'trycloudflare.com' in line:
                import re
                match = re.search(r'https://[^\s]*trycloudflare\.com', line)
                if match:
                    public_url = match.group(0)
                    break

    tunnel_thread = threading.Thread(target=read_tunnel, daemon=True)
    tunnel_thread.start()
    tunnel_thread.join(timeout=15)

    if public_url:
        print(f'Public URL: {public_url}')
    else:
        print('Cloudflare Tunnel started, but URL was not captured yet.')
else:
    print('Cloudflare Tunnel binary not found. API is running locally only.')

print('='*70)